# Summary_Day15_online.ipynb  
## 사용자 데이터 분류 · Custom Dataset · ImageFolder · Transfer Learning · ResNet18 Fine-tuning

이번 15강은 **내가 직접 준비한 이미지 데이터로 분류 모델을 만드는 방법**을 정리하는 강의다.

13~14강에서는 CIFAR-10 같은 정해진 데이터셋과 사전학습 모델을 사용했다.  
15강에서는 실제 프로젝트처럼 폴더에 직접 들어 있는 이미지 파일을 PyTorch Dataset으로 바꾸고, 사전학습 모델을 연결하는 흐름을 다룬다.

강의 핵심 흐름은 다음이다.

```text
문제 정의
→ 사용자 이미지 데이터 준비
→ ImageFolder가 요구하는 폴더 구조 이해
→ train / val 폴더와 class 폴더 구성
→ transforms로 전처리와 데이터 증강 정의
→ ImageFolder로 Dataset 생성
→ DataLoader로 mini-batch 공급
→ VGG19-BN fine-tuning
→ VGG19-BN transfer learning
→ dog vs wolf 사용자 데이터 분류
→ ResNet18 fine-tuning
→ learning rate scheduler
→ Early Stopping
→ 학습 곡선, 혼동 행렬, 샘플 예측 시각화
```

이 파일은 **인터넷 가능 버전**이다.  
강의 원본처럼 `hymenoptera_data.zip`, `dog_wolf.zip`, CIFAR-10, 사전학습 가중치 다운로드가 가능하다는 전제로 작성했다.

> 필기 포인트:  
> 실무에서는 데이터가 이미 Tensor로 정리되어 있지 않다.  
> 보통 jpg/png 이미지가 폴더에 들어 있고, 이 폴더 구조를 `ImageFolder`로 Dataset으로 바꾸는 흐름이 매우 중요하다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. 사용자가 직접 모은 이미지 데이터를 PyTorch Dataset으로 바꾸는 방법을 이해한다.
2. `ImageFolder`가 폴더 이름을 label로 인식하는 방식을 익힌다.
3. `transforms`에서 학습용 전처리와 검증용 전처리를 다르게 구성하는 이유를 이해한다.
4. VGG19-BN으로 ants vs bees를 fine-tuning한다.
5. VGG19-BN으로 ants vs bees를 transfer learning 방식으로 학습한다.
6. dog vs wolf처럼 매우 작은 사용자 데이터셋에서 transfer learning을 적용한다.
7. ResNet18 fine-tuning 코드에서 scheduler, early stopping, confusion matrix까지 실전 학습 루프를 정리한다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torchvision import models
```

- `torch`: Tensor 계산과 GPU 사용에 필요하다.
- `nn`: `Linear`, `CrossEntropyLoss` 같은 신경망 구성 요소를 만든다.
- `optim`: SGD 같은 Optimizer를 만든다.
- `transforms`: 이미지 전처리와 데이터 증강을 순서대로 묶는다.
- `datasets.ImageFolder`: 폴더 구조를 이미지 Dataset으로 바꾼다.
- `DataLoader`: Dataset을 batch 단위로 공급한다.
- `models`: VGG19-BN, ResNet18 같은 사전학습 모델을 불러온다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision import models

from sklearn.metrics import confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

## 3. device 설정과 시드 고정

### 함수 사용법

```python
torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

- GPU가 있으면 `cuda`를 사용한다.
- GPU가 없으면 `cpu`를 사용한다.

```python
torch.manual_seed(seed)
np.random.seed(seed)
```

- 난수를 고정해서 결과를 어느 정도 재현 가능하게 만든다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def torch_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

torch_seed(42)

print("device:", device)

## 4. ImageFolder 핵심 개념

`ImageFolder`는 폴더 이름을 class label로 인식한다.

필수 구조는 다음이다.

```text
root/
  train/
    ants/
      image001.jpg
      image002.jpg
    bees/
      image001.jpg
      image002.jpg
  val/
    ants/
      image001.jpg
    bees/
      image001.jpg
```

여기서 `ants`, `bees`가 class 이름이 된다.  
`ImageFolder(train_dir)`을 실행하면 이미지를 읽고, class 이름을 자동으로 숫자 label로 바꾼다.

> 시험 포인트:  
> `ImageFolder`는 파일명보다 폴더명을 label로 사용한다.

## 5. hymenoptera 데이터 다운로드와 압축 해제

강의 첫 번째 사용자 데이터 예제는 개미와 벌을 분류하는 `hymenoptera_data`다.

### 함수 사용법

```python
urllib.request.urlretrieve(url, zip_path)
zipfile.ZipFile(zip_path).extractall(...)
```

- `urlretrieve`: 인터넷 URL에서 zip 파일을 다운로드한다.
- `ZipFile.extractall`: zip 파일을 압축 해제한다.

In [ ]:
data_url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
zip_path = Path("hymenoptera_data.zip")
data_dir = Path("hymenoptera_data")

if not zip_path.exists():
    urllib.request.urlretrieve(data_url, zip_path)

if not data_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(".")

print("data_dir exists:", data_dir.exists())

for root, dirs, files in os.walk(data_dir):
    level = root.replace(str(data_dir), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    if level >= 2:
        continue

## 6. 학습용 transform과 검증용 transform 정의

학습 데이터에는 데이터 증강을 넣고, 검증 데이터에는 랜덤 증강을 넣지 않는다.

### 검증용 흐름

```text
Resize(256)
→ CenterCrop(224)
→ ToTensor
→ Normalize
```

### 학습용 흐름

```text
RandomResizedCrop(224)
→ RandomHorizontalFlip
→ ToTensor
→ RandomErasing
→ Normalize
```

> 주의:  
> `RandomErasing`은 Tensor에 적용되는 증강이라 `ToTensor()` 뒤에 와야 한다.

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.5,
        scale=(0.02, 0.33),
        ratio=(0.3, 3.3),
        value=0,
        inplace=False
    ),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("train_transform:")
print(train_transform)

print("\ntest_transform:")
print(test_transform)

## 7. ImageFolder로 Dataset 만들기

### 함수 사용법

```python
datasets.ImageFolder(root, transform=transform)
```

- `root`: class 폴더들이 들어 있는 상위 폴더다.
- `transform`: 이미지마다 적용할 전처리다.
- class 이름은 폴더 이름에서 자동으로 만들어진다.

In [ ]:
train_dir = data_dir / "train"
val_dir = data_dir / "val"

train_data = datasets.ImageFolder(train_dir, transform=train_transform)
train_data_no_aug = datasets.ImageFolder(train_dir, transform=test_transform)
val_data = datasets.ImageFolder(val_dir, transform=test_transform)

classes = train_data.classes

print("classes:", classes)
print("class_to_idx:", train_data.class_to_idx)
print("train_data:", len(train_data))
print("train_data_no_aug:", len(train_data_no_aug))
print("val_data:", len(val_data))

출력 해석:

- `classes`는 폴더 이름에서 자동으로 만들어진 class 이름이다.
- `class_to_idx`는 class 이름과 숫자 label의 매핑이다.
- `train_data`는 증강이 적용되는 학습용 Dataset이다.
- `train_data_no_aug`는 이미지 출력용이다.
- `val_data`는 검증용 Dataset이다.

## 8. DataLoader 만들기

### 함수 사용법

```python
DataLoader(dataset, batch_size=10, shuffle=True)
```

- `batch_size`: 한 번에 꺼낼 이미지 개수다.
- `shuffle=True`: 매 epoch마다 순서를 섞는다.
- 학습 데이터에는 `shuffle=True`를 쓴다.
- 검증 데이터에는 보통 `shuffle=False`를 쓴다.

In [ ]:
batch_size = 10

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

train_loader_preview = DataLoader(train_data_no_aug, batch_size=50, shuffle=True)
val_loader_preview = DataLoader(val_data, batch_size=50, shuffle=False)

images, labels = next(iter(train_loader))

print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("labels:", labels[:10])

## 9. 이미지 출력 함수

정규화된 이미지를 다시 `[0, 1]` 범위로 복원해서 출력한다.

In [ ]:
def show_images_labels(loader, classes, model=None, device=None, n_show=20):
    images, labels = next(iter(loader))

    predicted = None

    if model is not None:
        model.eval()
        with torch.no_grad():
            outputs = model(images.to(device))
            predicted = torch.max(outputs, 1)[1].cpu()

    n_show = min(n_show, len(images))

    plt.figure(figsize=(12, 5))

    for i in range(n_show):
        plt.subplot(2, 10, i + 1)

        img = images[i].numpy().transpose((1, 2, 0))
        img = (img + 1) / 2
        img = np.clip(img, 0, 1)

        true_name = classes[labels[i].item()]

        if predicted is None:
            title = true_name
            color = "black"
        else:
            pred_name = classes[predicted[i].item()]
            title = f"{true_name}\n→ {pred_name}"
            color = "blue" if predicted[i].item() == labels[i].item() else "red"

        plt.imshow(img)
        plt.title(title, fontsize=8, color=color)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_images_labels(val_loader_preview, classes, model=None, device=None, n_show=20)

## 10. 공통 학습 함수 만들기

강의 원본에서는 `pythonlibs.torch_lib1`의 `fit`, `evaluate_history`, `show_images_labels`를 사용한다.  
여기서는 노트북 하나만으로 실행되도록 같은 흐름의 함수를 직접 만든다.

In [ ]:
def train_one_epoch_simple(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        pred = torch.max(outputs, 1)[1]

        total_loss += loss.item() * labels.size(0)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate_simple(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            pred = torch.max(outputs, 1)[1]

            total_loss += loss.item() * labels.size(0)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
def fit_simple(model, optimizer, criterion, num_epochs, train_loader, val_loader, device):
    history = []

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch_simple(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_acc = evaluate_simple(
            model,
            val_loader,
            criterion,
            device
        )

        history.append([epoch + 1, train_loss, train_acc, val_loss, val_acc])

        print(
            f"epoch {epoch + 1} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    return np.array(history)


def evaluate_history(history, title="Learning Curve"):
    plt.plot(history[:, 0], history[:, 1], label="train loss")
    plt.plot(history[:, 0], history[:, 3], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title + " Loss")
    plt.legend()
    plt.show()

    plt.plot(history[:, 0], history[:, 2], label="train acc")
    plt.plot(history[:, 0], history[:, 4], label="val acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(title + " Accuracy")
    plt.legend()
    plt.show()

    print("최종 val accuracy:", history[-1, 4])

## 11. VGG19-BN Fine-tuning 모델 만들기

Fine-tuning은 사전학습된 모델의 **전체 파라미터를 업데이트**하는 방식이다.

강의에서는 VGG19-BN을 불러오고 마지막 classifier를 2개 class에 맞게 바꾼다.

### 함수 사용법

```python
models.vgg19_bn(pretrained=True)
net.classifier[6] = nn.Linear(in_features, 2)
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
```

- `classifier[6]`: VGG19-BN의 마지막 Linear layer다.
- `out_features=2`: ants/bees 두 class 분류다.
- `net.parameters()`: 전체 파라미터를 optimizer에 넘긴다.

In [ ]:
def load_vgg19_bn_pretrained():
    try:
        weights = models.VGG19_BN_Weights.DEFAULT
        net = models.vgg19_bn(weights=weights)
    except Exception:
        net = models.vgg19_bn(pretrained=True)
    return net


def build_vgg19_finetune(num_classes=2):
    net = load_vgg19_bn_pretrained()

    in_features = net.classifier[6].in_features
    net.classifier[6] = nn.Linear(in_features, num_classes)

    net.avgpool = nn.Identity()

    return net.to(device)


RUN_VGG_FINETUNE = False

if RUN_VGG_FINETUNE:
    torch_seed(42)

    net_ft = build_vgg19_finetune(num_classes=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net_ft.parameters(), lr=0.001, momentum=0.9)

    history_ft = fit_simple(
        net_ft,
        optimizer,
        criterion,
        num_epochs=5,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device
    )

    evaluate_history(history_ft, title="VGG19-BN Fine-tuning")
else:
    print("VGG19-BN fine-tuning은 무거워서 기본 실행에서는 건너뛴다.")
    print("실행하려면 RUN_VGG_FINETUNE = True로 바꾼다.")

## 12. VGG19-BN Transfer Learning 모델 만들기

Transfer Learning은 사전학습 모델의 대부분을 동결하고 마지막 분류기만 학습하는 방식이다.

### 핵심 코드

```python
for param in net.parameters():
    param.requires_grad = False

net.classifier[6] = nn.Linear(in_features, 2)
optimizer = optim.SGD(net.classifier[6].parameters(), lr=0.001, momentum=0.9)
```

- `requires_grad=False`: 기존 가중치를 동결한다.
- 새로 만든 classifier는 기본적으로 학습 가능하다.
- optimizer에는 마지막 layer의 파라미터만 넣는다.

In [ ]:
def build_vgg19_transfer(num_classes=2):
    net = load_vgg19_bn_pretrained()

    for param in net.parameters():
        param.requires_grad = False

    in_features = net.classifier[6].in_features
    net.classifier[6] = nn.Linear(in_features, num_classes)

    net.avgpool = nn.Identity()

    return net.to(device)


RUN_VGG_TRANSFER = False

if RUN_VGG_TRANSFER:
    torch_seed(42)

    net_tl = build_vgg19_transfer(num_classes=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net_tl.classifier[6].parameters(), lr=0.001, momentum=0.9)

    history_tl = fit_simple(
        net_tl,
        optimizer,
        criterion,
        num_epochs=5,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device
    )

    evaluate_history(history_tl, title="VGG19-BN Transfer Learning")
else:
    print("VGG19-BN transfer learning은 무거워서 기본 실행에서는 건너뛴다.")
    print("실행하려면 RUN_VGG_TRANSFER = True로 바꾼다.")

## 13. Fine-tuning과 Transfer Learning 비교

| 구분 | Fine-tuning | Transfer Learning |
|---|---|---|
| 학습 범위 | 전체 파라미터 | 마지막 분류기 중심 |
| 학습 시간 | 길다 | 짧다 |
| 데이터 요구량 | 더 많다 | 적은 데이터에도 가능 |
| 과적합 위험 | 상대적으로 큼 | 상대적으로 작음 |
| 강의 예시 | VGG19-BN 전체 업데이트 | VGG19-BN feature extractor 동결 |

강의에서는 ants/bees 예제에서 두 방식이 비슷한 검증 정확도를 보였다.  
데이터가 적을수록 보통 transfer learning이 더 안전하다.

## 14. 사용자 정의 dog vs wolf 데이터 다운로드

두 번째 사용자 데이터 예제는 dog vs wolf 분류다.  
데이터가 매우 적고 두 class가 비슷하게 생겨서 더 어려운 문제다.

강의에서는 `dog_wolf.zip`을 받아서 다음 구조로 사용한다.

```text
dog_wolf/
  train/
    dog/
    wolf/
  test/
    dog/
    wolf/
```

In [ ]:
dog_wolf_url = "https://github.com/makaishi2/pythonlibs/raw/main/images/dog_wolf.zip"
dog_wolf_zip = Path("dog_wolf.zip")
dog_wolf_dir = Path("dog_wolf")

if not dog_wolf_zip.exists():
    urllib.request.urlretrieve(dog_wolf_url, dog_wolf_zip)

if not dog_wolf_dir.exists():
    with zipfile.ZipFile(dog_wolf_zip, "r") as zip_ref:
        zip_ref.extractall(".")

print("dog_wolf exists:", dog_wolf_dir.exists())

for root, dirs, files in os.walk(dog_wolf_dir):
    level = root.replace(str(dog_wolf_dir), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    if level >= 2:
        continue

## 15. dog vs wolf transform과 Dataset

dog/wolf 이미지는 매우 적기 때문에 학습 데이터에 증강을 넣는다.

원본 코드에서는 `Normalize(0.5, 0.5)`처럼 작성되어 있지만, RGB 이미지에서는 보통 채널별 tuple을 쓰는 것이 안전하다.

```python
Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
```

In [ ]:
dog_test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dog_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.5,
        scale=(0.02, 0.33),
        ratio=(0.3, 3.3),
        value=0,
        inplace=False
    ),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dog_train_dir = dog_wolf_dir / "train"
dog_test_dir = dog_wolf_dir / "test"

dog_train_data = datasets.ImageFolder(dog_train_dir, transform=dog_train_transform)
dog_train_data_no_aug = datasets.ImageFolder(dog_train_dir, transform=dog_test_transform)
dog_test_data = datasets.ImageFolder(dog_test_dir, transform=dog_test_transform)

dog_classes = dog_train_data.classes

print("dog_classes:", dog_classes)
print("train:", len(dog_train_data))
print("test:", len(dog_test_data))

In [ ]:
dog_batch_size = 5

dog_train_loader = DataLoader(dog_train_data, batch_size=dog_batch_size, shuffle=True)
dog_train_loader_preview = DataLoader(dog_train_data_no_aug, batch_size=dog_batch_size, shuffle=True)
dog_test_loader = DataLoader(dog_test_data, batch_size=dog_batch_size, shuffle=False)

show_images_labels(dog_train_loader_preview, dog_classes, model=None, device=None, n_show=10)

## 16. dog vs wolf Transfer Learning

dog/wolf는 데이터가 매우 적기 때문에 transfer learning 방식이 적합하다.

```text
VGG19-BN pretrained
→ 전체 parameter freeze
→ classifier[6]을 dog/wolf 2 class로 교체
→ classifier[6]만 optimizer에 전달
```

In [ ]:
RUN_DOG_WOLF = False

if RUN_DOG_WOLF:
    torch_seed(42)

    net_dog = build_vgg19_transfer(num_classes=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net_dog.classifier[6].parameters(), lr=0.001, momentum=0.9)

    history_dog = fit_simple(
        net_dog,
        optimizer,
        criterion,
        num_epochs=10,
        train_loader=dog_train_loader,
        val_loader=dog_test_loader,
        device=device
    )

    evaluate_history(history_dog, title="Dog vs Wolf Transfer Learning")
    show_images_labels(dog_test_loader, dog_classes, net_dog, device, n_show=10)
else:
    print("dog vs wolf 학습은 기본 실행에서는 건너뛴다.")
    print("실행하려면 RUN_DOG_WOLF = True로 바꾼다.")

## 17. ResNet18 fine-tuning 실습 개요

15강 두 번째 노트북은 ResNet18을 사용해 CIFAR-10을 fine-tuning하는 실전형 코드다.

추가로 들어간 요소는 다음이다.

```text
ResNet18 pretrained
→ fc를 10 class로 교체
→ SGD with momentum
→ weight_decay
→ CosineAnnealingLR
→ EarlyStopping
→ 학습 곡선 시각화
→ Confusion Matrix
→ Classification Report
→ 샘플 예측 시각화
```

## 18. ResNet18용 transform과 CIFAR-10 DataLoader

훈련 데이터에는 증강을 넣고, 검증 데이터에는 랜덤 증강을 넣지 않는다.

### 훈련용

```text
Resize(128)
→ RandomCrop(112)
→ RandomHorizontalFlip
→ ColorJitter
→ ToTensor
→ Normalize(ImageNet mean/std)
```

### 검증용

```text
Resize(112)
→ ToTensor
→ Normalize(ImageNet mean/std)
```

In [ ]:
def get_resnet_transforms():
    train_transform = transforms.Compose([
        transforms.Resize(128),
        transforms.RandomCrop(112),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    val_transform = transforms.Compose([
        transforms.Resize(112),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, val_transform


def load_cifar10_data(batch_size=64):
    train_transform, val_transform = get_resnet_transforms()

    train_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=True,
        download=True,
        transform=train_transform
    )

    val_dataset = torchvision.datasets.CIFAR10(
        root="./data",
        train=False,
        download=True,
        transform=val_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader

print("ResNet18용 transform 준비 완료")

## 19. ResNet18 모델 생성 함수

ResNet18은 ImageNet 기준 1000 class를 분류하도록 만들어져 있다.  
CIFAR-10은 10 class이므로 마지막 `fc`를 교체한다.

In [ ]:
def create_resnet18_model(num_classes=10, pretrained=True):
    if pretrained:
        try:
            weights = models.ResNet18_Weights.IMAGENET1K_V1
            model = models.resnet18(weights=weights)
        except Exception:
            model = models.resnet18(pretrained=True)
    else:
        model = models.resnet18(weights=None)

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)

    return model.to(device)

model_resnet = create_resnet18_model(num_classes=10, pretrained=True)

total_params = sum(p.numel() for p in model_resnet.parameters())
trainable_params = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)

print("total params:", f"{total_params:,}")
print("trainable params:", f"{trainable_params:,}")
print(model_resnet.fc)

## 20. 학습 설정: Loss, Optimizer, Scheduler

### 함수 사용법

```python
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
```

- `CrossEntropyLoss`: 다중 분류 손실함수다.
- `SGD with momentum`: 이전 이동 방향을 반영해 학습한다.
- `weight_decay`: L2 정규화 역할을 하며 과적합을 줄이는 데 도움을 준다.
- `CosineAnnealingLR`: learning rate를 cosine 곡선처럼 부드럽게 줄인다.

In [ ]:
def setup_resnet_training(model, lr=0.001, momentum=0.9, num_epochs=20):
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=momentum,
        weight_decay=1e-4
    )

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6
    )

    return criterion, optimizer, scheduler

num_epochs = 20
criterion_resnet, optimizer_resnet, scheduler_resnet = setup_resnet_training(
    model_resnet,
    lr=0.001,
    momentum=0.9,
    num_epochs=num_epochs
)

print("초기 learning rate:", optimizer_resnet.param_groups[0]["lr"])
print("scheduler:", scheduler_resnet)

## 21. EarlyStopping 클래스

Early Stopping은 검증 손실이 더 이상 개선되지 않으면 학습을 멈추는 방법이다.

### 클래스 사용법

```python
early_stopping = EarlyStopping(patience=5, delta=0.001, path="resnet18_best.pth")
early_stopping(val_loss, model)
```

- `patience`: 개선이 없어도 기다릴 epoch 수다.
- `delta`: 개선으로 인정할 최소 변화량이다.
- `path`: 가장 좋은 모델을 저장할 파일명이다.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.0, path="best_model.pth"):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)

        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} / {self.patience}")

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        print(f"검증 손실 감소 ({self.val_loss_min:.4f} --> {val_loss:.4f}). 모델 저장")
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

early_stopping = EarlyStopping(patience=5, delta=0.001, path="resnet18_best.pth")

print("EarlyStopping 준비 완료")

## 22. ResNet18 학습 함수와 검증 함수

학습 함수는 다음 순서다.

```text
model.train()
→ batch를 device로 이동
→ optimizer.zero_grad()
→ outputs = model(inputs)
→ loss = criterion(outputs, labels)
→ loss.backward()
→ optimizer.step()
→ loss, accuracy 계산
```

검증 함수는 다음 순서다.

```text
model.eval()
→ torch.no_grad()
→ forward만 수행
→ loss, accuracy 계산
```

In [ ]:
def train_one_epoch_resnet(model, train_loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc


def validate_resnet(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

## 23. ResNet18 전체 학습 실행

전체 학습은 시간이 오래 걸릴 수 있어 기본 실행은 꺼 둔다.

실행하려면 다음 값을 바꾼다.

```python
RUN_RESNET_TRAINING = True
```

In [ ]:
RUN_RESNET_TRAINING = False

if RUN_RESNET_TRAINING:
    train_loader_resnet, val_loader_resnet = load_cifar10_data(batch_size=64)

    history_resnet = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": []
    }

    start_time = time.time()

    for epoch in range(num_epochs):
        print(f"[{epoch + 1}/{num_epochs}]")

        current_lr = optimizer_resnet.param_groups[0]["lr"]

        train_loss, train_acc = train_one_epoch_resnet(
            model_resnet,
            train_loader_resnet,
            criterion_resnet,
            optimizer_resnet,
            device
        )

        val_loss, val_acc = validate_resnet(
            model_resnet,
            val_loader_resnet,
            criterion_resnet,
            device
        )

        scheduler_resnet.step()

        history_resnet["train_loss"].append(train_loss)
        history_resnet["train_acc"].append(train_acc)
        history_resnet["val_loss"].append(val_loss)
        history_resnet["val_acc"].append(val_acc)
        history_resnet["lr"].append(current_lr)

        print(f"훈련 손실: {train_loss:.4f}, 훈련 정확도: {train_acc:.2f}%")
        print(f"검증 손실: {val_loss:.4f}, 검증 정확도: {val_acc:.2f}%")

        early_stopping(val_loss, model_resnet)

        if early_stopping.early_stop:
            print("Early stopping")
            break

    elapsed_time = time.time() - start_time
    print(f"총 소요 시간: {elapsed_time / 60:.2f}분")

    model_resnet.load_state_dict(torch.load("resnet18_best.pth", map_location=device))
else:
    print("ResNet18 전체 학습은 기본 실행에서 건너뛴다.")
    print("실행하려면 RUN_RESNET_TRAINING = True로 바꾼다.")

## 24. 학습 곡선 시각화 함수

학습 결과는 세 가지 그래프로 본다.

```text
Loss Curve
Accuracy Curve
Learning Rate Schedule
```

Learning Rate는 아주 작은 값까지 내려가므로 log scale로 보는 것이 편하다.

In [ ]:
def plot_history_resnet(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs, history["train_loss"], "b-o", label="Train Loss", linewidth=2)
    axes[0].plot(epochs, history["val_loss"], "r-s", label="Val Loss", linewidth=2)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss Curve")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], "b-o", label="Train Acc", linewidth=2)
    axes[1].plot(epochs, history["val_acc"], "r-s", label="Val Acc", linewidth=2)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("Accuracy Curve")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, history["lr"], "g-^", linewidth=2)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("Learning Rate Schedule")
    axes[2].set_yscale("log")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 25. Confusion Matrix와 Classification Report

혼동 행렬은 어떤 class를 어떤 class로 헷갈렸는지 보여준다.

### 함수 사용법

```python
confusion_matrix(y_true, y_pred)
classification_report(y_true, y_pred, target_names=class_names)
```

- `confusion_matrix`: 정답 class와 예측 class의 교차표다.
- `classification_report`: precision, recall, f1-score를 class별로 보여준다.

In [ ]:
def get_predictions(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


def plot_confusion_matrix_matplotlib(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)

    for (i, j), value in np.ndenumerate(cm):
        if value > 0:
            plt.text(j, i, str(value), ha="center", va="center", fontsize=8)

    plt.colorbar()
    plt.tight_layout()
    plt.show()

    class_accuracy = cm.diagonal() / np.maximum(cm.sum(axis=1), 1) * 100

    print("클래스별 정확도")
    print("=" * 40)

    for name, acc in zip(class_names, class_accuracy):
        print(f"{name:12s}: {acc:6.2f}%")

## 26. 샘플 예측 시각화 함수

정답과 예측을 함께 보여준다.

```text
맞으면 파란색
틀리면 빨간색
```

In [ ]:
def visualize_predictions(model, loader, class_names, device, num_images=16):
    model.eval()

    images, labels = next(iter(loader))

    images = images[:num_images]
    labels = labels[:num_images]

    with torch.no_grad():
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs, 1)
        predicted = predicted.cpu()

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    images = images * std + mean
    images = torch.clamp(images, 0, 1)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.ravel()

    for idx in range(num_images):
        img = images[idx].permute(1, 2, 0).numpy()

        true_label = class_names[labels[idx]]
        pred_label = class_names[predicted[idx]]

        color = "blue" if labels[idx] == predicted[idx] else "red"

        axes[idx].imshow(img)
        axes[idx].set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=10)
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()

## 27. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `Custom Dataset` | 사용자가 직접 준비한 데이터 | 폴더 이미지, CSV 등 |
| `ImageFolder` | 폴더 구조 기반 이미지 Dataset | `datasets.ImageFolder(root)` |
| `root` | 데이터 상위 폴더 | class 폴더가 들어 있는 위치 |
| `class_to_idx` | class 이름과 숫자 label 매핑 | 자동 생성 |
| `transform` | 이미지 전처리 | Resize, Crop, ToTensor 등 |
| `RandomResizedCrop` | 랜덤 crop 후 resize | 학습 증강 |
| `CenterCrop` | 중앙 crop | 검증 전처리 |
| `RandomErasing` | 일부 영역 지우기 | Tensor 뒤에 적용 |
| `DataLoader` | batch 공급 도구 | `DataLoader(dataset, batch_size)` |
| `Fine-tuning` | 전체 파라미터 미세 조정 | `optimizer = SGD(net.parameters())` |
| `Transfer Learning` | 기존 가중치 동결 후 classifier 학습 | `requires_grad=False` |
| `requires_grad` | gradient 계산 여부 | `False`면 동결 |
| `classifier[6]` | VGG19-BN 마지막 layer | 2 class로 교체 |
| `fc` | ResNet18 마지막 layer | 10 class로 교체 |
| `CosineAnnealingLR` | cosine 형태 LR scheduler | 학습률을 부드럽게 감소 |
| `EarlyStopping` | 검증 손실 개선이 없으면 중단 | 과적합 방지 |
| `Confusion Matrix` | class별 예측 오류 표 | 어떤 class를 헷갈렸는지 확인 |
| `classification_report` | class별 지표표 | precision, recall, f1 |

## 28. 시험용 요약

```text
15강 핵심 = 직접 준비한 이미지 폴더를 ImageFolder로 Dataset으로 만들고, 사전학습 모델을 붙여 분류한다
```

꼭 기억할 것:

- 실무에서는 이미지 파일을 직접 폴더에 넣고 분류하는 경우가 많다.
- `ImageFolder`는 폴더 이름을 class label로 인식한다.
- 폴더 구조는 `train/class_name/image.jpg`, `val/class_name/image.jpg` 형태가 기본이다.
- 학습 transform에는 데이터 증강을 넣는다.
- 검증 transform에는 랜덤 증강을 넣지 않는다.
- `ToTensor()`는 이미지를 Tensor로 바꾼다.
- `RandomErasing`은 Tensor 뒤에 적용해야 한다.
- `DataLoader`는 Dataset을 batch 단위로 공급한다.
- Fine-tuning은 전체 파라미터를 업데이트한다.
- Transfer Learning은 대부분의 파라미터를 동결하고 마지막 분류기만 학습한다.
- 데이터가 적으면 Transfer Learning이 더 안전한 경우가 많다.
- VGG19-BN의 마지막 layer는 `classifier[6]`이다.
- ResNet18의 마지막 layer는 `fc`다.
- ResNet18 CIFAR-10 fine-tuning에서는 `fc`를 10 class로 바꾼다.
- `CosineAnnealingLR`은 learning rate를 부드럽게 줄인다.
- Early Stopping은 검증 손실이 개선되지 않을 때 학습을 멈춘다.
- Confusion Matrix는 어떤 class끼리 헷갈렸는지 보여준다.